<a href="https://colab.research.google.com/github/SamDerden/Fraud_Detection_with-XAI_and_LLM/blob/main/test_antifraud_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pytest
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, accuracy_score, classification_report
)


# ============================================================
# Фикстуры
# ============================================================

@pytest.fixture(scope="module")
def data():
    """Синтетические данные транзакций (дисбаланс ~15%)."""
    X, y = make_classification(
        n_samples=3000,
        n_features=12,
        n_informative=8,
        n_redundant=2,
        weights=[0.85, 0.15],
        flip_y=0.02,
        random_state=42,
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
    }


@pytest.fixture(scope="module")
def model(data):
    """Обученная модель (лучший конфиг из экспериментов)."""
    clf = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1,
    )
    clf.fit(data["X_train"], data["y_train"])
    return clf


# ============================================================
# Тесты
# ============================================================

def test_model_can_predict(model, data):
    """Модель должна успешно делать предсказания."""
    preds = model.predict(data["X_test"])
    assert preds is not None
    assert len(preds) == len(data["X_test"])


def test_prediction_shape_and_values(model, data):
    """Предсказания — только 0 или 1, правильная форма."""
    preds = model.predict(data["X_test"])
    assert preds.shape == (len(data["X_test"]),)
    assert set(np.unique(preds)).issubset({0, 1})


def test_predict_proba_range(model, data):
    """Вероятности в [0, 1] и суммируются в 1."""
    proba = model.predict_proba(data["X_test"])
    assert proba.shape == (len(data["X_test"]), 2)
    assert np.all(proba >= 0.0) and np.all(proba <= 1.0)
    assert np.allclose(proba.sum(axis=1), 1.0, atol=1e-5)


def test_model_not_trivial(model, data):
    """Модель не должна предсказывать только один класс."""
    preds = model.predict(data["X_test"])
    unique = np.unique(preds)
    assert len(unique) == 2, "Модель предсказывает только один класс"


def test_minimum_quality_metrics(model, data):
    """Минимальные требования к качеству на тесте."""
    preds = model.predict(data["X_test"])
    proba = model.predict_proba(data["X_test"])[:, 1]

    f1 = f1_score(data["y_test"], preds)
    roc_auc = roc_auc_score(data["y_test"], proba)
    precision = precision_score(data["y_test"], preds)
    recall = recall_score(data["y_test"], preds)

    assert f1 >= 0.70, f"F1 слишком низкий: {f1:.3f}"
    assert roc_auc >= 0.85, f"ROC-AUC слишком низкий: {roc_auc:.3f}"
    assert precision >= 0.80, f"Precision слишком низкий: {precision:.3f}"
    assert recall >= 0.55, f"Recall слишком низкий: {recall:.3f}"


def test_handles_extreme_feature_values(model, data):
    """Модель не падает на экстремальных значениях признаков."""
    X_extreme = data["X_test"].copy()
    X_extreme[0, 0] = 1e6
    X_extreme[1, 1] = -1e5
    X_extreme[2, :] = 0.0

    preds = model.predict(X_extreme[:5])
    proba = model.predict_proba(X_extreme[:5])

    assert preds.shape == (5,)
    assert proba.shape == (5, 2)
    assert np.all(np.isfinite(proba))


def test_prediction_consistency(model, data):
    """Повторный вызов predict даёт тот же результат."""
    preds1 = model.predict(data["X_test"][:100])
    preds2 = model.predict(data["X_test"][:100])
    assert np.array_equal(preds1, preds2)


def test_feature_importance_exists(model):
    """Есть feature_importances_ (нужно для XAI)."""
    assert hasattr(model, "feature_importances_")
    importances = model.feature_importances_
    assert len(importances) == 12
    assert np.all(importances >= 0)
    assert np.isclose(importances.sum(), 1.0, atol=1e-5)


# ============================================================
# Подробный вывод при запуске через python
# ============================================================

def print_detailed_report():
    """Печатает читаемый отчёт о качестве модели."""
    print("=" * 60)
    print("ОТЧЁТ ПО ТЕСТИРОВАНИЮ МОДЕЛИ АНТИФРОДА")
    print("=" * 60)

    # Данные
    X, y = make_classification(
        n_samples=3000, n_features=12, n_informative=8, n_redundant=2,
        weights=[0.85, 0.15], flip_y=0.02, random_state=42
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    print(f"\nДанные:")
    print(f"  Train: {X_train.shape[0]} объектов")
    print(f"  Test:  {X_test.shape[0]} объектов")
    print(f"  Доля фрода в train: {y_train.mean():.1%}")

    # Модель
    model = RandomForestClassifier(
        n_estimators=100, max_depth=None, min_samples_leaf=1,
        random_state=42, n_jobs=-1
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    # Метрики
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    roc_auc = roc_auc_score(y_test, proba)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)

    print(f"\nМетрики на тесте:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  F1-score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc_auc:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")

    print(f"\nClassification report:")
    print(classification_report(y_test, preds, target_names=["legit", "fraud"], digits=3))

    # Проверки
    print("Результаты проверок:")
    checks = [
        ("Предсказания работают", True),
        ("Форма и значения корректны (0/1)", set(np.unique(preds)).issubset({0, 1})),
        ("Вероятности в [0, 1]", np.all((proba >= 0) & (proba <= 1))),
        ("Модель не trivial (оба класса)", len(np.unique(preds)) == 2),
        ("F1 >= 0.70", f1 >= 0.70),
        ("ROC-AUC >= 0.85", roc_auc >= 0.85),
        ("Precision >= 0.80", precision >= 0.80),
        ("Recall >= 0.55", recall >= 0.55),
        ("Есть feature_importances_", hasattr(model, "feature_importances_")),
    ]

    all_ok = True
    for name, ok in checks:
        status = "PASS" if ok else "FAIL"
        icon = "OK" if ok else "X "
        print(f"  [{icon}] {name}: {status}")
        if not ok:
            all_ok = False

    print("\n" + "=" * 60)
    if all_ok:
        print("ИТОГ: все проверки пройдены")
    else:
        print("ИТОГ: есть проваленные проверки — см. выше")
    print("=" * 60)

    print("\nГлавное слабое место: Recall ≈ {:.2f} — модель пропускает часть фрода.".format(recall))
    print("Рекомендация: class_weight='balanced' или подбор threshold.")


if __name__ == "__main__":
    print_detailed_report()

ОТЧЁТ ПО ТЕСТИРОВАНИЮ МОДЕЛИ АНТИФРОДА

Данные:
  Train: 2250 объектов
  Test:  750 объектов
  Доля фрода в train: 15.8%

Метрики на тесте:
  Accuracy:  0.9227
  F1-score:  0.7010
  ROC-AUC:   0.9420
  Precision: 0.9067
  Recall:    0.5714

Classification report:
              precision    recall  f1-score   support

       legit      0.924     0.989     0.956       631
       fraud      0.907     0.571     0.701       119

    accuracy                          0.923       750
   macro avg      0.916     0.780     0.828       750
weighted avg      0.922     0.923     0.915       750

Результаты проверок:
  [OK] Предсказания работают: PASS
  [OK] Форма и значения корректны (0/1): PASS
  [OK] Вероятности в [0, 1]: PASS
  [OK] Модель не trivial (оба класса): PASS
  [OK] F1 >= 0.70: PASS
  [OK] ROC-AUC >= 0.85: PASS
  [OK] Precision >= 0.80: PASS
  [OK] Recall >= 0.55: PASS
  [OK] Есть feature_importances_: PASS

ИТОГ: все проверки пройдены

Главное слабое место: Recall ≈ 0.57 — модель про